[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xylem-lab/GEO-ANOM-dashboard/blob/experiment/retraining-data-2026-11-01/notebooks/09_training_data.ipynb)

# 09 — Training data for fine-tuning the poultry-house model

**Why.** The published model (Robinson et al. 2022, trained on Delmarva houses and older photos) finds about two-thirds of the hand-traced houses on the 2023 NAIP photos on its own (80% with the outline, SAM and re-toning fixes) but about half of the houses it misses on 2023 are found on the 2017–2021 photos. The 2023 photos are much darker (RGB ~71 vs ~100). Notebook 10 fine-tunes the same model; this notebook builds what it learns from.

**What.** 512 m windows of NAIP (1 m, RGB + near-infrared) from **2017, 2021 and 2023**, each with a mask: 1 = poultry house, 0 = not, 255 = ignore.
- **House windows** are centred on groups of USGS hand-traced houses (Soroka & Duren 2016/17, every Delmarva house, registered or not) so every house is whole in at least one window.
- **Empty windows** are random land squares with no house within 300 m (half as many as house windows, drawn afresh each year).
- **Ignored:** a 2 m band along every traced roof edge (photos of different years sit a few metres apart), no-data pixels, and houses on our map that aren't in the tracing until notebook 07 settles them.
- **From notebook 07:** houses every reviewer confirmed become houses (on 2023); detections every reviewer rejected become "not a house" examples (warehouses, roads, the airport). Re-run this notebook after labelling: it adds those windows and rebuilds the masks; nothing already downloaded is fetched again.

**Split by whole county** so the test is on places the model never trained on: **test = Wicomico + Queen Anne's**, **validation = Dorchester** (picks when to stop training), **train = the other six**. Notebook 10 reports the held-out counties with the same scoring as notebook 05.

**Outputs.** `Task1_results/training/v1/` (`plan.geojson`, `images.jsonl`, `<split>/<year>/images|masks/*.npz`, `training_meta.json`). ~4,400 windows, ~3.7 GB. Locations of houses, no farm names.
**Runtime.** Downloads are the slow part (~1–2 h in Colab; resumable — Run all again after a crash). Masks take minutes.

In [ ]:
# ── Setup: the same cell works in Google Colab and locally ──────────────────────
import os, sys, json, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "experiment/retraining-data-2026-11-01"   # branch, tag or commit to run; the exact commit is recorded in run_meta.json

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                     # results go to the shared drive's Task1_results
    ROOT = Path("/content/GEO-ANOM-dashboard")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/xylem-lab/GEO-ANOM-dashboard", str(ROOT)], check=True)
    # always bring the copy up to date (a reused Colab session keeps an old copy)
    head = lambda: subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout
    before = head()
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=True)
    remote = f"origin/{CODE_VERSION}"
    target = remote if subprocess.run(["git", "-C", str(ROOT), "rev-parse", "-q", "--verify", remote],
                                      capture_output=True).returncode == 0 else CODE_VERSION
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", target], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
    if head() != before and "geo_anom" in sys.modules:   # new code, but the old version is already loaded
        raise RuntimeError("Code updated. Runtime -> Restart session, then Run all again.")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
# reload edited geo_anom/ code automatically, without restarting the kernel
if "google.colab" not in sys.modules:   # Colab's IPython can't load autoreload on Python 3.12+ (no `imp`)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

from geo_anom.task1 import env, tiles, region, viz
from geo_anom.task1 import county as C
from geo_anom.task1.kmz import write_kmz
from geo_anom.task1.pipeline import run_pipeline
from geo_anom.task1.evaluate import evaluate_run, load_ground_truth
from geo_anom.task1 import review as R
from geo_anom.task1 import training as T
from geo_anom.task1.pipeline import git_commit
print("running in", "Colab" if IN_COLAB else "local", "| results ->", env.results_root())

## Settings

In [ ]:
YEARS = T.YEARS                    # ["2017", "2021", "2023"]
SPLITS = T.SPLITS                  # test: Wicomico, Queen Anne's | val: Dorchester | train: the rest
COUNTIES = C.EASTERN_SHORE
NEG_RATIO = 0.5                    # empty windows per house window, per county and year
MAX_WORKERS = 12
OUT = env.results_root() / "training" / "v1"
REVIEW_DIR = env.results_root() / "review"
FINAL_MAP = env.results_root() / "final_eastern_shore" / "eastern_shore_houses_final.geojson"
OUT.mkdir(parents=True, exist_ok=True)
print("training data ->", OUT)

## 1. Labels

The USGS tracing, our final Eastern Shore map (2,823 houses: houses not in the tracing are new since 2016/17 or mistakes, so they're ignored until checked), and whatever notebook 07 has settled so far.

In [ ]:
gt = load_ground_truth()
houses = gpd.read_file(FINAL_MAP) if FINAL_MAP.exists() else None
if houses is None:
    print("final map not found -- using notebook 05's detections instead")
    parts = [C.load_county_run(C.county_out(n))[2] for n in COUNTIES if (C.county_out(n) / "buildings.geojson").exists()]
    houses = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=parts[0].crs)
confirmed, rejected, review_labels = T.review_polygons(REVIEW_DIR, C.all_counties_in_order())
print(f"07 labels: {len(review_labels)} | confirmed new houses {len(confirmed)} | rejected {len(rejected)}")
labels = T.label_layer(gt, houses, confirmed, rejected)
display(labels.source.value_counts().rename("polygons").to_frame()
        .assign(meaning=lambda d: [{"traced": "house, every year", "new_unchecked": "ignored",
                                     "new_confirmed": "house in 2023, ignored before", "rejected": "not a house"}[s] for s in d.index]))

## 2. Which windows

Made once and saved; later runs load it and only add windows for 07 results that aren't already covered.

In [ ]:
PLAN = OUT / "plan.geojson"
if PLAN.exists():
    plan = gpd.read_file(PLAN)
    plan["year"] = plan["year"].astype(str)
    print("loaded saved plan:", len(plan), "windows")
else:
    plan = T.plan_windows(COUNTIES, labels, years=YEARS, neg_ratio=NEG_RATIO, splits=SPLITS)
    plan.to_file(PLAN, driver="GeoJSON")
    print("new plan:", len(plan), "windows")
extra = T.extra_windows(plan, labels, years=YEARS, splits=SPLITS)
if len(extra):
    plan = gpd.GeoDataFrame(pd.concat([plan, extra.to_crs(plan.crs)], ignore_index=True), crs=plan.crs)
    plan.to_file(PLAN, driver="GeoJSON")
    print(f"added {len(extra)} windows from notebook 07 labels")
tab = plan[plan.year == YEARS[-1]].pivot_table(index=["split", "county"], columns="kind", values="window_id",
                                                 aggfunc="count", fill_value=0)
display(Markdown("**Windows per year** (each is downloaded for every year)")); display(tab)
fig, ax = plt.subplots(figsize=(7, 8))
region.md_counties()[lambda d: d.NAME.isin(COUNTIES)].to_crs(T.UTM).plot(ax=ax, color="0.93", edgecolor="0.6")
colors = {"train": "#2b7a3d", "val": "#c8902e", "test": "#b03a2e"}
for s, g in plan[plan.year == YEARS[-1]].groupby("split"):
    g.centroid.plot(ax=ax, markersize=3, color=colors[s], label=s)
ax.legend(); ax.set_axis_off(); ax.set_title("Window centres by split"); plt.show()

## 3. Download the photos (slow, resumable)

In [ ]:
from tqdm.auto import tqdm
log = T.download_windows(plan, OUT, max_workers=MAX_WORKERS, progress=tqdm)
got = plan.merge(log[["window_id", "status"]], on="window_id", how="left").fillna({"status": "not yet"})
display(got.pivot_table(index="year", columns="status", values="window_id", aggfunc="count", fill_value=0))

## 4. Masks (fast; re-run after 07)

In [ ]:
stats = T.build_masks(plan, labels, OUT, progress=tqdm)
s = stats.groupby(["split", "year"]).agg(windows=("window_id", "size"), house_px=("house_px", "sum"),
                                         ignore_px=("ignore_px", "sum"), px=("px", "sum"))
s["house %"] = (100 * s.house_px / s.px).round(2); s["ignored %"] = (100 * s.ignore_px / s.px).round(2)
display(s[["windows", "house %", "ignored %"]])

## 5. Look at it

Same windows in each year, mask over the photo (red = house, yellow = ignored). Check that the red sits on the roofs in every year, and how much darker 2023 is.

In [ ]:
import random
def show(ids, title=""):
    fig, ax = plt.subplots(len(ids), len(YEARS), figsize=(4.2 * len(YEARS), 4.2 * len(ids)), squeeze=False)
    for i, cell in enumerate(ids):
        for j, y in enumerate(YEARS):
            r = plan[(plan.cell == cell) & (plan.year == y)]
            a = ax[i, j]; a.axis("off")
            if not len(r): continue
            ip, mp = T._paths(OUT, r.iloc[0])
            if not (ip.exists() and mp.exists()): continue
            img, m = np.load(ip)["img"], np.load(mp)["mask"]
            rgb = img[..., :3].astype(float) / 255
            ov = rgb.copy(); ov[m == 1] = ov[m == 1] * 0.5 + np.array([1, .2, .2]) * .5; ov[m == 255] = [1, 1, 0]
            a.imshow(ov); a.set_title(f"{y} · {r.iloc[0].split} · RGB {255 * rgb[img.any(2)].mean():.0f}", fontsize=9)
    plt.tight_layout(); plt.show()

random.seed(0)
pos = sorted(plan[plan.kind == "positive"].cell.unique())
show(random.sample(pos, 4))

**Brightness by year** (mean RGB of each window's land pixels) — the gap the colour changes in training are meant to cover.

In [ ]:
rows = []
for _, r in plan[plan.kind == "positive"].groupby("year").head(150).iterrows():
    ip, _ = T._paths(OUT, r)
    if ip.exists():
        img = np.load(ip)["img"]; v = img[img.any(2)][:, :3]
        rows.append({"year": r.year, "rgb_mean": v.mean()})
b = pd.DataFrame(rows)
display(b.groupby("year").rgb_mean.describe()[["count", "mean", "25%", "50%", "75%"]].round(1))
dates = log.explode("dates").assign(year=lambda d: d.dates.str[:4]).groupby("year").dates.value_counts()
display(Markdown("**NAIP flight dates used**")); display(dates.rename("windows").to_frame().head(30))

**What training will see:** six random training crops (256 px) of one window — rotated, flipped, rescaled 0.8–1.25×, and brighter or darker (down to ~40% darker, like the dark 2023 flights). Most crops are placed on a house.

In [ ]:
r = plan[(plan.kind == "positive") & (plan.split == "train")].iloc[0]
ip, mp = T._paths(OUT, r)
img, m = np.load(ip)["img"], np.load(mp)["mask"]
rng = np.random.default_rng(1)
fig, ax = plt.subplots(1, 6, figsize=(20, 3.6))
for a in ax:
    x, mm = T.augment(img, m, rng)
    a.imshow(x[..., :3]); a.contour(mm == 1, [0.5], colors="r", linewidths=0.8); a.axis("off")
    a.set_title(f"RGB {x[..., :3][x.any(2)].mean():.0f}", fontsize=9)
plt.show()
ds = T.ChipDataset(OUT, plan, "train")
x, y = ds[0]
print(f"train windows ready: {len(ds)} | one sample: {tuple(x.shape)} {x.dtype}, mask {tuple(y.shape)} values {sorted(set(y.unique().tolist()))}")
for sp in ["val", "test"]:
    print(f"{sp} windows ready: {len(T.ChipDataset(OUT, plan, sp, train=False))}")

## 6. Save what this data is

In [ ]:
meta = {
    "made_at": pd.Timestamp.now().isoformat(timespec="seconds"), "git_commit": git_commit(),
    "years": YEARS, "splits": SPLITS, "counties": COUNTIES, "neg_ratio": NEG_RATIO,
    "window_m": T.WINDOW_M, "crop_px": T.CROP_PX, "border_m": T.BORDER_M, "rules": T.RULES,
    "resolution_m": tiles.TARGET_RESOLUTION_M,
    "label_sources": {"tracing": env.GROUND_TRUTH["source"], "final_map": str(FINAL_MAP.name),
                      "review_labels": int(len(review_labels)), "confirmed_new": int(len(confirmed)),
                      "rejected": int(len(rejected))},
    "polygons": labels.source.value_counts().to_dict(),
    "windows": plan.groupby(["split", "year"]).size().rename("n").reset_index().to_dict("records"),
    "downloaded": log.status.value_counts().to_dict() if len(log) else {},
    "mask_share": {f"{k[0]}/{k[1]}": {"house_%": float(v["house %"]), "ignored_%": float(v["ignored %"])}
                   for k, v in s.iterrows()},
    "base_model": env.CHECKPOINT["source"],
}
(OUT / "training_meta.json").write_text(json.dumps(meta, indent=2, default=str))
print("saved", OUT / "training_meta.json")

## What this shows

*(Write after the run: windows per split and year, how many downloads failed, house and ignored share per split, whether the masks sit on the roofs in every year, brightness by year, how many 07 labels went in.)*